In [ ]:
import nltk
import pycrfsuite
from sklearn_crfsuite import metrics

# Load data
nltk.download('treebank')
corpus = nltk.corpus.treebank.tagged_sents()

# Feature function
def word_features(sentence, i):
    word = sentence[i][0] if isinstance(sentence[i], tuple) else sentence[i]
    return {
        'word': word,
        'is_first': i == 0,
        'is_last': i == len(sentence)-1,
        'prefix': word[:2],
        'suffix': word[-2:],
    }

# Prepare data
X, y = [], []
for sentence in corpus:
    X_sent, y_sent = [], []
    for i in range(len(sentence)):
        X_sent.append(word_features(sentence, i))
        y_sent.append(sentence[i][1])
    X.append(X_sent)
    y.append(y_sent)

# Split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train CRF
trainer = pycrfsuite.Trainer(verbose=False)
for x, y_ in zip(X_train, y_train):
    trainer.append(x, y_)

trainer.set_params({
    'c1': 1.0,
    'c2': 0.001,
    'max_iterations': 50
})

trainer.train('pos.crfsuite')

# Load model
tagger = pycrfsuite.Tagger()
tagger.open('pos.crfsuite')

# Test accuracy
y_pred = [tagger.tag(x) for x in X_test]
print("Accuracy:", metrics.flat_accuracy_score(y_test, y_pred))

# Tag new sentence
sentence = "Geeksforgeeks is a best platform for students".split()
features = [word_features(sentence, i) for i in range(len(sentence))]
tags = tagger.tag(features)

print(list(zip(sentence, tags)))

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


Accuracy: 0.9487000349318828
[('Geeksforgeeks', 'NNS'), ('is', 'VBZ'), ('a', 'DT'), ('best', 'JJS'), ('platform', 'NN'), ('for', 'IN'), ('students', 'NNS')]


In [ ]:
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics

# Load dataset
nltk.download('treebank')
corpus = nltk.corpus.treebank.tagged_sents()

# Feature function
def word_features(sentence, i):
    word = sentence[i][0]
    return {
        'word': word,
        'is_first': i == 0,
        'is_last': i == len(sentence)-1,
        'is_capitalized': word[0].upper() == word[0],
        'prefix': word[:2],
        'suffix': word[-2:],
        'prev_word': '' if i == 0 else sentence[i-1][0],
        'next_word': '' if i == len(sentence)-1 else sentence[i+1][0],
    }

# Prepare data
X = []
y = []

for sentence in corpus:
    X_sentence = []
    y_sentence = []
    for i in range(len(sentence)):
        X_sentence.append(word_features(sentence, i))
        y_sentence.append(sentence[i][1])
    X.append(X_sentence)
    y.append(y_sentence)

# Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train CRF
crf = sklearn_crfsuite.CRF(max_iterations=100)
crf.fit(X_train, y_train)

# Predict
y_pred = crf.predict(X_test)

# Accuracy
print("Accuracy:", metrics.flat_accuracy_score(y_test, y_pred))

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


Accuracy: 0.9502470183142872


In [ ]:
!pip install sklearn-crfsuite python-crfsuite
import nltk, pycrfsuite
from sklearn_crfsuite import metrics

nltk.download('treebank')
data = nltk.corpus.treebank.tagged_sents()

def features(sent, i):
    w = sent[i][0] if isinstance(sent[i], tuple) else sent[i]
    return {'word': w, 'prev': '' if i==0 else sent[i-1][0]}

X, y = [], []
for s in data:
    X.append([features(s,i) for i in range(len(s))])
    y.append([tag for _,tag in s])

split = int(0.8*len(X))
Xtr, Xte = X[:split], X[split:]
ytr, yte = y[:split], y[split:]

trainer = pycrfsuite.Trainer(verbose=False)
for x, t in zip(Xtr, ytr): trainer.append(x, t)
trainer.train('model.crf')

tagger = pycrfsuite.Tagger()
tagger.open('model.crf')

# Accuracy
pred = [tagger.tag(x) for x in Xte]
print("Acc:", metrics.flat_accuracy_score(yte, pred))

# New sentence
sent = "Geeksforgeeks is best".split()
feat = [features(sent,i) for i in range(len(sent))]
print(list(zip(sent, tagger.tag(feat))))

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.


Acc: 0.8892659314337042
[('Geeksforgeeks', 'NNP'), ('is', 'VBZ'), ('best', 'JJS')]
